# E12 Digit Recognizer：高维向量、降维直觉与错分分析

> Issue #9 | 本地 `sklearn.datasets.load_digits`（8×8）；Kaggle [Digit Recognizer](https://www.kaggle.com/competitions/digit-recognizer) 为 28×28，流程相同

## 你将学到
1. **图像即向量**：每张图拉平为 64 维；高维空间稀疏，直接逐列直方图 EDA 低效 → **先看图**
2. **PCA / t-SNE 直觉**：PCA 保留全局方差；t-SNE 强调局部邻域（非线性，计算贵，QUICK 下对子集演示）
3. **易混数字对**：3/5、8/5 等在低分辨率下形态接近
4. **分层划分** + QuickBaseline
5. **错分可视化**：从混淆矩阵定位再抽查图像

## 提交
`Label` 列 + `prediction_type='label'`


In [ ]:
QUICK = True
RANDOM_STATE = 42

# 一键就绪：关警告 + 打印环境 + 设种子（勿再叠 JE.no_warning / filterwarnings）
from wanglaoshi.runtime import ready
ready(seed=RANDOM_STATE)


## 1. 加载 digits


In [ ]:
from sklearn.datasets import load_digits
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

digits = load_digits()
X = digits.data
y = digits.target
print('X', X.shape, '— each row is 8x8=64 pixels flattened')
print('class counts:\n', pd.Series(y).value_counts().sort_index())


## 2. 像素 EDA：均值图与每类一例


In [ ]:
mean_img = X.mean(axis=0).reshape(8, 8)
plt.figure(figsize=(3, 3))
plt.imshow(mean_img, cmap='gray')
plt.title('Mean digit (all classes)')
plt.axis('off')
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in zip(range(10), axes.ravel()):
    idx = np.where(y == digit)[0][0]
    ax.imshow(X[idx].reshape(8, 8), cmap='gray')
    ax.set_title(f'digit {digit}')
    ax.axis('off')
plt.suptitle('One example per class')
plt.tight_layout()
plt.show()


## 3. 分层 train/test


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print('train bincount', np.bincount(ytr))
print('test  bincount', np.bincount(yte))


## 4. PCA：二维投影与方差解释

[PCA](https://scikit-learn.org/stable/modules/decomposition.html#pca) 在正交方向上最大化方差；2D 投影便于观察类簇是否可分。


In [ ]:
from sklearn.decomposition import PCA

pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
Z = pca2.fit_transform(Xtr)
plt.figure(figsize=(6, 5))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=ytr, cmap='tab10', s=12, alpha=0.85)
plt.colorbar(sc)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title(f'PCA 2D on train (var={pca2.explained_variance_ratio_.sum():.1%})')
plt.grid(True, alpha=0.3)
plt.show()

pca95 = PCA(n_components=0.95, random_state=RANDOM_STATE).fit(Xtr)
print('n_components for 95% variance:', pca95.n_components_)


### 4.1 t-SNE 直觉（QUICK：子集 300 条）


In [ ]:
from sklearn.manifold import TSNE

n_sub = min(300, len(Xtr))
rng = np.random.RandomState(RANDOM_STATE)
sub_idx = rng.choice(len(Xtr), size=n_sub, replace=False)
X_sub, y_sub = Xtr[sub_idx], ytr[sub_idx]
tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30, max_iter=300 if QUICK else 1000)
Zt = tsne.fit_transform(X_sub)
plt.figure(figsize=(6, 5))
plt.scatter(Zt[:, 0], Zt[:, 1], c=y_sub, cmap='tab10', s=15)
plt.title('t-SNE 2D (subset) — local neighborhood emphasis')
plt.colorbar()
plt.show()


## 5. QuickBaseline


In [ ]:
from wanglaoshi import compete as CT

Xtr_df = pd.DataFrame(Xtr)
Xte_df = pd.DataFrame(Xte)
baseline = CT.QuickBaseline()
results = baseline.run_all_models(
    Xtr_df, pd.Series(ytr), Xte_df, pd.Series(yte), task_type='classification'
)
comparison = baseline.compare_models(results, top_n=5)
comparison


## 6. 分类报告与混淆矩阵


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best = comparison.iloc[0]['Model']
pred = np.asarray(results[best]['predictions']).astype(int)
print(classification_report(yte, pred))
cm = confusion_matrix(yte, pred)
print(cm)


### 6.1 易混对：非对角最大项


In [ ]:
cm_off = cm.copy().astype(int)
np.fill_diagonal(cm_off, 0)
i, j = np.unravel_index(cm_off.argmax(), cm_off.shape)
print(f'most confused: true {i} predicted as {j} (count={cm_off[i, j]})')
for pair in [(3, 5), (5, 8), (8, 3)]:
    a, b = pair
    if a < cm.shape[0] and b < cm.shape[1]:
        print(f'  {a}->{b}: {cm[a, b]} errors')


## 7. 错分样本可视化


In [ ]:
wrong_idx = np.where(pred != yte)[0]
print('n_errors', len(wrong_idx))
show = wrong_idx[: min(12, len(wrong_idx))]
ncols = max(1, len(show))
fig, axes = plt.subplots(1, ncols, figsize=(1.4 * ncols, 2.2))
if ncols == 1:
    axes = [axes]
for ax, wi in zip(axes, show):
    ax.imshow(Xte[wi].reshape(8, 8), cmap='gray')
    ax.set_title(f't={yte[wi]} p={pred[wi]}', fontsize=9)
    ax.axis('off')
plt.suptitle('Misclassified digits')
plt.tight_layout()
plt.show()


### 7.1 分层 CV 基线稳定性


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', KNeighborsClassifier(n_neighbors=5)),
])
cv = StratifiedKFold(n_splits=3 if QUICK else 5, shuffle=True, random_state=RANDOM_STATE)
acc = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy')
print(f'KNN stratified CV accuracy: {acc.mean():.3f} ± {acc.std():.3f}')


### 7.2 每类平均字形（centroid 图像）


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 4.5))
for d, ax in zip(range(10), axes.ravel()):
    centroid = X[y == d].mean(axis=0).reshape(8, 8)
    ax.imshow(centroid, cmap='gray')
    ax.set_title(f'avg {d}')
    ax.axis('off')
plt.suptitle('Class mean images (centroids)')
plt.tight_layout()
plt.show()


### 7.3 每类准确率与混淆热力图


In [ ]:
import seaborn as sns

per_class_acc = []
for d in range(10):
    m = yte == d
    per_class_acc.append((pred[m] == d).mean() if m.any() else np.nan)
print('per-class accuracy:', dict(enumerate(np.round(per_class_acc, 3))))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples')
plt.xlabel('predicted')
plt.ylabel('true')
plt.title('Digits confusion heatmap')
plt.tight_layout()
plt.show()


### 7.4 像素方差最高的维度（信息量大的像素）


In [ ]:
var_px = X.var(axis=0).reshape(8, 8)
plt.figure(figsize=(4, 4))
plt.imshow(var_px, cmap='hot')
plt.title('Per-pixel variance across dataset')
plt.colorbar(fraction=0.046)
plt.axis('off')
plt.show()
print('top-5 variable pixel indices:', np.argsort(X.var(axis=0))[-5:][::-1])


## 8. 模拟提交


In [ ]:
from pathlib import Path

out = Path('outputs/12_digit_recognizer')
out.mkdir(parents=True, exist_ok=True)
sample = pd.DataFrame({'ImageId': np.arange(1, len(pred) + 1), 'Label': yte})
sample.to_csv(out / 'sample_submission.csv', index=False)
gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    pred, str(out / 'sample_submission.csv'), str(out / 'submission.csv'), prediction_type='label'
)
print(gen.validate_submission(str(out / 'submission.csv'), str(out / 'sample_submission.csv')))
sub.head()


### 8.1 SVM 与 KNN 在 PCA 子空间的快速对比


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

n_pca = 20 if QUICK else 30
pca_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=n_pca, random_state=RANDOM_STATE)),
    ('clf', LinearSVC(random_state=RANDOM_STATE, max_iter=5000)),
])
pca_clf.fit(Xtr, ytr)
print('LinearSVC + PCA acc', accuracy_score(yte, pca_clf.predict(Xte)))
knn_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=n_pca, random_state=RANDOM_STATE)),
    ('clf', KNeighborsClassifier(n_neighbors=3)),
])
knn_pca.fit(Xtr, ytr)
print('KNN + PCA acc', accuracy_score(yte, knn_pca.predict(Xte)))


### 8.2 展示一组训练样本栅格（加深图像 EDA）


In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
ix = rng.choice(len(Xtr), size=min(32, len(Xtr)), replace=False)
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for ax, i in zip(axes.ravel(), ix):
    ax.imshow(Xtr[i].reshape(8, 8), cmap='gray')
    ax.set_title(str(ytr[i]), fontsize=8)
    ax.axis('off')
plt.suptitle('Random train batch')
plt.tight_layout()
plt.show()


## 9. 检查清单
- [x] 高维向量 + 图像 EDA
- [x] PCA 与 t-SNE 直觉
- [x] 混淆矩阵 + 3/5/8 易混讨论
- [x] 错分图 + 分层 CV
- [x] Label 提交
- [ ] 对易混对做数据增强或专用二分类器


## 附录 A：图像向量化后的「维度诅咒」

8×8 数字 → 64 维。许多维度是背景零；欧氏距离在高维会变钝。PCA 保留主要笔画能量，既可视化又降噪。

错分通常发生在 **形状相近** 的数字对：3/5/8、4/9。


In [ ]:
# 展示若干错分样本（若 Xte 为 DataFrame/ndarray）
try:
    import matplotlib.pyplot as plt
    Xte_arr = np.asarray(Xte)
    yte_arr = np.asarray(yte)
    pred_arr = np.asarray(pred)
    wrong = np.where(pred_arr != yte_arr)[0][:8]
    if len(wrong):
        fig, axes = plt.subplots(2, 4, figsize=(8, 4))
        for ax, i in zip(axes.ravel(), wrong):
            ax.imshow(Xte_arr[i].reshape(8, 8), cmap='gray')
            ax.set_title(f'true{yte_arr[i]} pred{pred_arr[i]}', fontsize=8)
            ax.axis('off')
        plt.suptitle('Misclassified digits'); plt.tight_layout(); plt.show()
    else:
        print('no errors on this split — try harder model or different seed')
except Exception as e:
    print('viz skipped', e)


## 附录 B：基线阶梯

1. 像素直接喂 Logistic / RF  
2. PCA(0.9 var) + Logistic  
3. 简单增强（可选，非本课强制）

每一步只改一个因素，才能归因。


In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
try:
    pipe = Pipeline([('pca', PCA(0.90, random_state=RANDOM_STATE)), ('clf', LogisticRegression(max_iter=1000))])
    cv = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)
    print('PCA+LR CV acc', cross_val_score(pipe, X, y, cv=cv).mean())
except Exception as e:
    print('ladder skipped', e)


## 附录 C：提交与评测一致性

Digits 本地模拟 Kaggle Digit Recognizer：ID + Label。验证指标必须来自 **有标签的 holdout**，测试文件无标签时禁止「本地刷测试分」。


## 深度阅读与作业（Digits）

### 必读观点
1. 高维欧氏距离退化；PCA 便于可视化与降噪。
2. 错分集中在形近数字对；误差分析重于刷 0.01 分。
3. 无标签测试集只用于生成提交。

### 书面作业
1. 展示至少 4 张错分图并猜测原因（笔画断裂/倾斜）。
2. 比较原始像素 vs PCA+LR 的 CV。
3. 完成一次坏提交校验（打乱 ID）。


## 课堂讨论记录区（请学生填写）

主题：12_digit_recognizer

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。


## 课堂讨论记录区（请学生填写）

主题：12_digit_recognizer

1. 本日最重要的一个概念：________________
2. 我曾犯过的一个泄露/划分错误：________________
3. 对应的 wanglaoshi API：________________
4. 若重做实验，我将固定的种子与协议：________________

（以下为延长阅读摘要，帮助达到课程阅读量）

数据分析正统流程始终是：**问题定义 → 数据质量 → 协议（划分/指标）→ 基线 → 诊断 → 交付**。任何跳过协议直接调参的行为，都会在换数据时暴露。本课数据集的经典地位，来自它能把上述某几环的错误放大到「肉眼可见」。请把本笔记与仓库 `docs/API_REFERENCE.md`、`book/lessons/` 精讲对照阅读。
